# NucleoScan fragment-emergence demo — Google Colab

This notebook demonstrates a simplified confidence-emergence scan using
ESMFold on the Trp-cage miniprotein (PDB: 1L2Y, 20 residues). It does not
simulate folding time or identify a transition-state folding nucleus.

**Before running:** enable GPU runtime via *Runtime → Change runtime type → T4 GPU*

**Reference:**
Lin et al. (2023) *Science* 379, 1123–1130. https://doi.org/10.1126/science.ade2574

## Step 1 — Clone repository and install dependencies

In [ ]:
import subprocess, sys

# Clone the pipeline repository
subprocess.run(
    ["git", "clone", "--depth", "1",
     "https://github.com/ahiriadil-lab/NucleoScan.git",
     "/content/NucleoScan"],
    check=True
)
%cd /content/NucleoScan

# Install PyTorch (Colab already has a compatible version)
# Install ESMFold and remaining dependencies
!pip install -q 'fair-esm[esmfold]'
!pip install -q -r requirements.txt

print('Installation complete.')

## Step 2 — Verify GPU and imports

In [ ]:
import torch
import esm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch  : {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('GPU      : not available — inference will be slow on CPU')

## Step 3 — Load ESMFold model

In [ ]:
print('Loading ESMFold v1 weights (~2.5 GB, first run only)...')
_model = esm.pretrained.esmfold_v1().eval()
if torch.cuda.is_available():
    _model = _model.cuda()
print('Model ready.')

## Step 4 — Define prediction helpers

In [ ]:
FLANK = 15  # flanking glycines on each side (reduces edge effects)


def predict_pdb(sequence: str, seed: int = 42) -> str:
    """Run ESMFold inference and return PDB string."""
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    flanked = 'G' * FLANK + sequence + 'G' * FLANK
    with torch.no_grad():
        pdb_str = _model.infer_pdb(flanked)
    return _strip_flanking(pdb_str, n_left=FLANK, frag_len=len(sequence))


def _strip_flanking(pdb_str: str, n_left: int, frag_len: int) -> str:
    """Remove flanking glycine residues from predicted PDB and renumber 1..frag_len."""
    lines = []
    for line in pdb_str.split('\n'):
        if line.startswith('ATOM'):
            try:
                res_num = int(line[22:26])
                if n_left < res_num <= n_left + frag_len:
                    new_num = res_num - n_left
                    line = line[:22] + f'{new_num:4d}' + line[26:]
                    lines.append(line)
            except ValueError:
                pass
        elif not line.startswith('HETATM'):
            lines.append(line)
    return '\n'.join(lines)


def mean_plddt(pdb_str: str) -> float:
    """Extract mean pLDDT from the B-factor column of a PDB string."""
    values = []
    for line in pdb_str.split('\n'):
        if line.startswith('ATOM'):
            try:
                values.append(float(line[60:66]))
            except ValueError:
                pass
    return float(np.mean(values)) if values else 0.0


print('Helpers defined.')

## Step 5 — Define protein and run fragment predictions

The pipeline grows N-terminal fragments of increasing length L and measures
the ESMFold confidence (mean pLDDT) for each fragment. The per-residue
delta δ(i) = pLDDT(L) − pLDDT(L−1) measures an association between prefix
extension and model confidence. It is not a causal or temporal folding step.

Change `SEQUENCE` below to run on a different protein.

In [ ]:
# Trp-cage miniprotein (PDB: 1L2Y, 20 residues)
PROTEIN_NAME = 'Trp-cage (1L2Y)'
SEQUENCE     = 'NLYIQWLKDGGPSSGRPPPS'

N_SEEDS   = 1    # ESMFold inference is deterministic for an unchanged input
MIN_LEN   = 5    # minimum fragment length

print(f'Protein : {PROTEIN_NAME}')
print(f'Sequence: {SEQUENCE}  ({len(SEQUENCE)} aa)')
print(f'Fragments: lengths {MIN_LEN}..{len(SEQUENCE)}, one prediction each')
print()

records = []
n_fragments = len(SEQUENCE) - MIN_LEN + 1

for L in range(MIN_LEN, len(SEQUENCE) + 1):
    fragment = SEQUENCE[:L]
    plddt_values = []
    for seed in range(42, 42 + N_SEEDS):
        pdb_str = predict_pdb(fragment, seed=seed)
        plddt_values.append(mean_plddt(pdb_str))
    records.append({'length': L, 'plddt': np.mean(plddt_values)})
    print(f'  L={L:2d}  fragment={fragment[:12]:12s}...  pLDDT={np.mean(plddt_values):.1f}')

frag_df = pd.DataFrame(records)
print(f'\nDone. {len(frag_df)} fragments predicted.')

## Step 6 — Compute confidence-emergence candidates

In [ ]:
# Delta score: δ(i) = pLDDT(L=i) - pLDDT(L=i-1)
plddt = frag_df['plddt'].values
delta = np.diff(plddt, prepend=plddt[0])  # δ(MIN_LEN) = 0 by convention

residues = np.arange(MIN_LEN, len(SEQUENCE) + 1)
res_df = pd.DataFrame({'residue': residues, 'delta': delta})

# Adaptive threshold: mean + 1 * std
mu, sigma = delta.mean(), delta.std()
threshold = mu + 1.0 * sigma
res_df['candidate'] = res_df['delta'] > threshold

candidate_residues = res_df[res_df['candidate']]['residue'].tolist()
print(f'Threshold: {threshold:.3f}  (μ={mu:.3f}, σ={sigma:.3f})')
print(f'High-delta candidates ({len(candidate_residues)}): {candidate_residues}')
print(f'Candidate fraction: {len(candidate_residues)/len(SEQUENCE)*100:.0f}%')

## Step 7 — Visualise results

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

# Panel A: pLDDT vs fragment length
axes[0].plot(frag_df['length'], frag_df['plddt'], 'o-', color='steelblue', lw=2, ms=5)
axes[0].set_ylabel('Mean pLDDT', fontsize=12)
axes[0].set_title(f'{PROTEIN_NAME} — Fragment pLDDT confidence', fontsize=13)
axes[0].axhline(70, color='gray', ls='--', lw=1, label='pLDDT=70')
axes[0].legend(fontsize=10)
axes[0].set_ylim(0, 100)
axes[0].grid(True, alpha=0.3)

# Panel B: per-residue confidence delta with candidates coloured
colors = ['tomato' if n else 'steelblue' for n in res_df['candidate']]
axes[1].bar(res_df['residue'], res_df['delta'], color=colors, edgecolor='none')
axes[1].axhline(threshold, color='black', ls='--', lw=1.5, label=f'Threshold = {threshold:.3f}')
axes[1].set_xlabel('Residue position', fontsize=12)
axes[1].set_ylabel('Δ mean pLDDT', fontsize=12)
axes[1].set_title('Confidence-emergence delta  (red = candidate)', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

# Annotate thresholded candidates
for _, row in res_df[res_df['candidate']].iterrows():
    aa = SEQUENCE[int(row['residue']) - 1]
    axes[1].annotate(f"{aa}{int(row['residue'])}",
                     xy=(row['residue'], row['delta']),
                     xytext=(0, 6), textcoords='offset points',
                     ha='center', fontsize=8, color='darkred')

plt.tight_layout()
plt.savefig('/content/confidence_emergence_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to /content/confidence_emergence_plot.png')

## Step 8 — Save results

In [ ]:
frag_df.to_csv('/content/fragment_scores.csv', index=False)
res_df.to_csv('/content/residue_scores.csv', index=False)

print('Results saved:')
print('  /content/fragment_scores.csv  — pLDDT per fragment length')
print('  /content/residue_scores.csv   — per-residue deltas and candidate flags')
print('  /content/confidence_emergence_plot.png — figure')
print()
print('Download from the Colab file browser (folder icon on the left).')

## Notes

This notebook uses a simplified pLDDT-only delta score for demonstration.
The full pipeline (`run_dataset.py`) fuses three structural quality metrics
(backbone RMSD, Q-value, hydrophobic Rg) and supports four scoring methods
(`marginal`, `combined`, `ab_initio`, `unified`), exploratory comparison
with audited φ-values, sensitivity analysis, and publication-quality figures.

**pLDDT interpretation (ESMFold):**
- ≥ 90: very high confidence (well-structured)
- 70–90: high confidence
- 50–70: medium confidence
- < 50: low confidence (likely disordered)

Trp6 and the polyproline segment are useful structural landmarks, but
agreement with those positions would not by itself validate a folding nucleus.